In [18]:
import os
import sys
if os.getcwd().endswith("notebook"):
    os.chdir("..")
sys.path.append("src")

import time
import cv2
import numpy as np
import pandas as pd
from utils_cv import baca, kontras_bercak

# ==========================================
# KODE 4.1: Verifikasi Konvolusi Manual
# ==========================================
def konvolusi(img, k):
    kh, kw = k.shape
    ph, pw = kh // 2, kw // 2
    pad = np.pad(img.astype(float), ((ph, ph), (pw, pw)), mode="edge")
    out = np.zeros(img.shape)
    for y in range(img.shape[0]):
        for x in range(img.shape[1]):
            out[y, x] = np.sum(pad[y:y+kh, x:x+kw] * k)
    return np.clip(np.round(out), 0, 255).astype(np.uint8)

def konvolusi_sejati(img, k):
    return konvolusi(img, np.flip(k))

nama = "teh_041.png"
bersih = cv2.cvtColor(baca(f"data/sintetis/bersih/{nama}"), cv2.COLOR_BGR2GRAY)
md = cv2.imread(f"data/sintetis/mask_daun/{nama}", 0)
mb = cv2.imread(f"data/sintetis/mask_bercak/{nama}", 0)

print("--- VERIFIKASI KONVOLUSI MANUAL VS FILTER2D ---")
for nm, k in {"rerata": np.ones((3, 3)) / 9, "asimetris": np.array([[1, 0, -1], [2, 0, -2], [1, 0, -1]], float)}.items():
    t0 = time.perf_counter()
    a = konvolusi(bersih, k)
    t_manual = time.perf_counter() - t0

    t0 = time.perf_counter()
    b = cv2.filter2D(bersih, -1, k, borderType=cv2.BORDER_REPLICATE)
    t_cv = time.perf_counter() - t0

    beda = np.abs(a.astype(int) - b.astype(int)).max()
    c = konvolusi_sejati(bersih, k)
    beda2 = np.abs(c.astype(int) - b.astype(int)).max()
    print(f"{nm:9s}: selisih manual vs filter2D={beda}, sejati vs filter2D={beda2}, waktu manual={t_manual:.3f} s, filter2D={t_cv:.5f} s")

# ==========================================
# KODE 4.2: Tabel Penapis x Derau x PSNR x Waktu x KB (Tabel 11)
# ==========================================
rng = np.random.default_rng(42)

def derau_gauss(img, sd):
    return np.clip(img + rng.normal(0, sd, img.shape), 0, 255).astype(np.uint8)

def derau_sp(img, p):
    out = img.copy()
    m = rng.random(img.shape)
    out[m < p/2] = 0
    out[m > 1 - p/2] = 255
    return out

def ukur_waktu(f, x, ulang=10):
    t = []
    for _ in range(ulang):
        t0 = time.perf_counter()
        f(x)
        t.append(time.perf_counter() - t0)
    return 1000 * np.mean(t)

derau = {
    "gauss_20": derau_gauss(bersih, 20),
    "sp_4%": derau_sp(bersih, 0.04)
}

penapis = {
    "rerata_3": lambda x: cv2.blur(x, (3, 3)),
    "rerata_5": lambda x: cv2.blur(x, (5, 5)),
    "gauss_5": lambda x: cv2.GaussianBlur(x, (5, 5), 1.0),
    "median_3": lambda x: cv2.medianBlur(x, 3),
    "median_5": lambda x: cv2.medianBlur(x, 5),
    "median_9": lambda x: cv2.medianBlur(x, 9)
}

baris = []
for dn, d in derau.items():
    baris.append([dn, "tanpa", cv2.PSNR(bersih, d), 0.0, kontras_bercak(d, md, mb)])
    for pn, f in penapis.items():
        out = f(d)
        baris.append([dn, pn, cv2.PSNR(bersih, out), ukur_waktu(f, d), kontras_bercak(out, md, mb)])

tabel = pd.DataFrame(baris, columns=["derau", "penapis", "PSNR_dB", "waktu_ms", "KB"]).round(2)
tabel.to_csv("hasil/b4_tabel_penapis.csv", index=False)

print("\n--- TABEL 11: RINGKASAN PERBANDINGAN PENAPIS ---")
print(tabel.to_string(index=False))

--- VERIFIKASI KONVOLUSI MANUAL VS FILTER2D ---
rerata   : selisih manual vs filter2D=0, sejati vs filter2D=0, waktu manual=1.959 s, filter2D=0.00905 s
asimetris: selisih manual vs filter2D=0, sejati vs filter2D=255, waktu manual=2.024 s, filter2D=0.00040 s

--- TABEL 11: RINGKASAN PERBANDINGAN PENAPIS ---
   derau  penapis  PSNR_dB  waktu_ms   KB
gauss_20    tanpa    22.10      0.00 1.68
gauss_20 rerata_3    31.40      0.16 3.83
gauss_20 rerata_5    34.41      0.16 4.63
gauss_20  gauss_5    32.53      0.30 4.11
gauss_20 median_3    29.72      0.10 3.41
gauss_20 median_5    33.32      0.73 4.43
gauss_20 median_9    35.20     15.91 5.14
   sp_4%    tanpa    19.26      0.00 1.25
   sp_4% rerata_3    28.50      0.13 3.15
   sp_4% rerata_5    31.82      0.21 4.10
   sp_4%  gauss_5    29.61      0.29 3.44
   sp_4% median_3    48.20      0.09 5.49
   sp_4% median_5    44.41      0.73 5.72
   sp_4% median_9    40.05     12.98 6.61
